# Product Extraction Debug

Step-by-step product data extraction with multi-strategy merge.

In [ ]:
# 1. SETUP
import sys
from pathlib import Path

backend_path = Path.cwd().parent
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

# Add prod_page_v2 for local imports
prod_path = Path.cwd()
if str(prod_path) not in sys.path:
    sys.path.insert(0, str(prod_path))

from dotenv import load_dotenv
load_dotenv(backend_path.parent / 'config' / '.env')

print(f"Backend: {backend_path}")
print(f"Prod page: {prod_path}")
print("Run next cell to start browser pool.")

In [ ]:
# 2. BROWSER POOL
from browser_pool import BrowserPool

# Cleanup existing
if 'pool' in globals():
    try:
        await pool.shutdown()
    except: pass

pool = BrowserPool(size=3, pages_per_recycle=20, headless=False)
await pool.start()

print(f"Browser pool ready ({pool.size} browsers).")

In [ ]:
# 3. RELOAD MODULES (run after code changes)
import importlib

modules_to_reload = [
    'extractor',
    'page_loader',
    'browser_pool',
    'llm_image_extractor',
    'strategies.shopify',
    'strategies.ld_json',
    'strategies.api_intercept',
    'strategies.llm_schema',
]

for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from extractor import ProductExtractor
from page_loader import load_page_on_existing, extract_gallery_images
from llm_image_extractor import extract_product_images_with_llm

# Test brands - add your brands/product URLs here
BRANDS = {
    "khaite": {
        "url": "https://khaite.com",
        "products": [
            "https://khaite.com/products/the-cleo-dress-in-black",
            "https://khaite.com/products/the-mael-jacket-in-nero",
        ]
    },
    "axel_arigato": {
        "url": "https://www.axelarigato.com",
        "products": [
            "https://www.axelarigato.com/us/clean-90-leather-white-23001",
            "https://www.axelarigato.com/us/marathon-runner-white-multi-93141",
        ]
    },
    "entire_studios": {
        "url": "https://www.entirestudios.com",
        "products": [
            # Add product URLs here
        ]
    },
    "eckhaus_latta": {
        "url": "https://www.eckhauslatta.com",
        "products": [
            # Add product URLs here
        ]
    },
}

print(f"Modules reloaded. {len(BRANDS)} brands available.")
print("Brands:", list(BRANDS.keys()))

In [ ]:
# 4. EXTRACT SINGLE PRODUCT (with discovery)
BRAND = "khaite"  # <-- Change this
PRODUCT_IDX = 0    # <-- Which product from the list

brand_data = BRANDS[BRAND]
url = brand_data["products"][PRODUCT_IDX]

print(f"\n{'='*60}")
print(f"Extracting: {BRAND}")
print(f"URL: {url}")
print(f"{'='*60}\n")

extractor = ProductExtractor()

# Run discovery on the single product
contributions, results, ground_truth, field_sources = await extractor.discover(url)

print(f"\n{'='*60}")
print(f"RESULTS: {BRAND}")
print(f"{'='*60}")
print(f"Strategies that worked: {len(contributions)}")
for c in contributions:
    print(f"  - {c.strategy.value}: {', '.join(sorted(c.fields))} (score: {c.score})")

if field_sources:
    print(f"\nField sources (GT-validated):")
    for field, strategy in sorted(field_sources.items()):
        print(f"  - {field} → {strategy}")

if ground_truth:
    print(f"\nGround truth:")
    print(f"  Name: {ground_truth.name}")
    print(f"  Price: {ground_truth.price} {ground_truth.currency}")
    print(f"  Variants: {len(ground_truth.variants) if ground_truth.variants else 0}")

In [ ]:
# 5. DISCOVER + VERIFY (full pipeline for 2 products)
BRAND = "khaite"  # <-- Change this

brand_data = BRANDS[BRAND]
products = brand_data["products"]

if len(products) < 2:
    print(f"Need at least 2 products for {BRAND}, got {len(products)}")
else:
    print(f"\n{'='*60}")
    print(f"Discovery + Verification: {BRAND}")
    print(f"{'='*60}\n")

    extractor = ProductExtractor()
    domain = BRAND.replace('_', '.')  # Approximate domain

    config = await extractor.discover_and_verify(
        domain=domain,
        product_urls=products[:2]
    )

    if config:
        print(f"\n✓ Config created for {domain}")
        print(f"  Strategies: {[c.strategy.value for c in config.contributions]}")
    else:
        print(f"\n✗ Discovery failed for {BRAND}")

In [ ]:
# 6. BATCH EXTRACT (using pool)
BRAND = "khaite"  # <-- Change this

brand_data = BRANDS[BRAND]
products = brand_data["products"]

print(f"\n{'='*60}")
print(f"Batch extraction: {BRAND} ({len(products)} products)")
print(f"{'='*60}\n")

extractor = ProductExtractor()
results = []

for i, url in enumerate(products):
    print(f"\n[{i+1}/{len(products)}] {url}")
    async with pool.acquire() as page:
        result = await extractor.extract_single_pooled(url, page)
        results.append(result)
        
        if result.success:
            p = result.product
            print(f"  ✓ {p.name[:40]}... | {p.price} {p.currency} | {len(p.images)} imgs | {len(p.variants)} variants")
        else:
            print(f"  ✗ {result.error}")

# Summary
success = sum(1 for r in results if r.success)
print(f"\n{'='*60}")
print(f"SUMMARY: {success}/{len(results)} successful")
print(f"{'='*60}")

In [ ]:
# 7. GALLERY IMAGE EXTRACTION TEST
BRAND = "khaite"  # <-- Change this
PRODUCT_IDX = 0

brand_data = BRANDS[BRAND]
url = brand_data["products"][PRODUCT_IDX]

print(f"\n{'='*60}")
print(f"Gallery extraction: {url}")
print(f"{'='*60}\n")

async with pool.acquire() as page:
    # Load page
    page_data = await load_page_on_existing(page, url, wait_time=3000)
    print(f"Page loaded: {len(page_data.html)} chars")
    
    # Get product name from page
    product_name = await page.evaluate("document.querySelector('h1')?.textContent?.trim() || ''")
    print(f"Product: {product_name}")
    
    # Extract images with LLM
    print(f"\nRunning LLM image extraction...")
    result = await extract_product_images_with_llm(page, product_name, url, max_images=50)
    
    print(f"\nResults:")
    print(f"  Indices: {result.get('product_image_indices', [])}")
    print(f"  Selector: {result.get('gallery_selector', 'N/A')}")
    print(f"  Selector valid: {result.get('selector_valid', False)}")
    print(f"  Selector count: {result.get('selector_count', 0)}")
    print(f"  Reasoning: {result.get('reasoning', 'N/A')}")
    
    # Show URLs
    urls = result.get('product_image_urls', [])
    print(f"\nProduct images ({len(urls)}):")
    for i, img_url in enumerate(urls[:10]):
        print(f"  {i+1}. {img_url[:80]}...")
    if len(urls) > 10:
        print(f"  ... and {len(urls) - 10} more")

In [ ]:
# 8. TEST GALLERY SELECTOR DIRECTLY
# Use this to test a specific CSS selector

BRAND = "khaite"  # <-- Change this
PRODUCT_IDX = 0
SELECTOR = ".product-gallery img"  # <-- Your selector here

brand_data = BRANDS[BRAND]
url = brand_data["products"][PRODUCT_IDX]

print(f"Testing selector: {SELECTOR}")
print(f"URL: {url}\n")

async with pool.acquire() as page:
    await load_page_on_existing(page, url, wait_time=3000)
    
    # Test selector
    gallery_config = {
        "image_selectors": [SELECTOR],
        "url_attribute": "src"
    }
    
    images = await extract_gallery_images(page, gallery_config)
    
    print(f"Found {len(images)} images:")
    for i, img in enumerate(images[:15]):
        print(f"  {i+1}. {img[:80]}...")
    if len(images) > 15:
        print(f"  ... and {len(images) - 15} more")

In [ ]:
# 9. SHOW MERGED RESULT (after single product extraction)
if 'results' in dir() and results:
    for i, r in enumerate(results):
        if r.success:
            p = r.product
            print(f"\n{'='*60}")
            print(f"Product {i+1}:")
            print(f"{'='*60}")
            print(f"Name: {p.name}")
            print(f"Price: {p.price} {p.currency}")
            print(f"Brand: {p.brand}")
            print(f"SKU: {p.sku}")
            print(f"Category: {p.category}")
            print(f"\nDescription ({len(p.description)} chars):")
            print(f"  {p.description[:200]}..." if len(p.description) > 200 else f"  {p.description}")
            print(f"\nImages ({len(p.images)}):")
            for j, img in enumerate(p.images[:5]):
                print(f"  {j+1}. {img[:70]}...")
            print(f"\nVariants ({len(p.variants)}):")
            for v in p.variants[:5]:
                avail = "✓" if v.available else "✗"
                print(f"  {avail} {v.size or '-'} / {v.color or '-'} | {v.sku}")
else:
    print("No results. Run extraction first.")

In [ ]:
# 10. TEST ALL BRANDS (batch)
all_results = {}

for brand, data in BRANDS.items():
    products = data.get("products", [])
    if not products:
        print(f"[{brand}] No products configured, skipping")
        continue
        
    print(f"\n{'='*60}")
    print(f"[{brand}] Testing {len(products)} products")
    print(f"{'='*60}")
    
    extractor = ProductExtractor()
    brand_results = []
    
    for i, url in enumerate(products):
        try:
            async with pool.acquire() as page:
                result = await extractor.extract_single_pooled(url, page)
                brand_results.append(result)
                
                if result.success:
                    p = result.product
                    print(f"  ✓ {p.name[:30]}... | {p.price} | {len(p.images)} imgs")
                else:
                    print(f"  ✗ {result.error}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
            brand_results.append(None)
    
    all_results[brand] = brand_results

# Summary
print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for brand, res in all_results.items():
    success = sum(1 for r in res if r and r.success)
    total = len(res)
    status = "✓" if success == total else "⚠" if success > 0 else "✗"
    print(f"{status} {brand}: {success}/{total} successful")

In [ ]:
# 11. CLEANUP
if 'pool' in globals():
    await pool.shutdown()
    print("Browser pool shut down.")